# はじめての PWS Cup 2026

配布された練習キットを使って、**加工（防御）と攻撃を1周ずつ実際に手を動かして体験する**ための
ノートブックです。所要 20〜30 分。

> **このノートブックは任意の教材です。**
> ルールと提出フォーマットの正本は **ルールブック**と **`participant_data/sample_submissions/`** です。
> ここに書いてあることとルールブックが食い違ったら、ルールブックが正しいと考えてください。
> 提出物は CSV とテキストファイルだけなので、**Python 以外の言語・ツールで作ってもかまいません**
> （自己採点器・検証器を動かすには Python か Docker が要ります）。

## 使い方

```sh
cd starter
uv sync --extra notebook
uv run jupyter lab ..
```

自己採点のセルだけ Docker を使います。Docker が無くても他のセルは動きます
（`python -m pwscup2026.kit.selfscore` でも同じ点数が出ます）。

**採点の中身をもっと詳しく知りたい場合は、`採点のしくみ.ipynb` を続けてどうぞ。**

## 0. 配布物を確認する

| 記号 | 意味 | 誰が持つか | 練習キットでの配布 |
| --- | --- | --- | --- |
| 母集団 | 日本全国を模した仮想の人口全体 | 事務局のみ | × |
| `A_bg` | 母集団のうち、どのチームのコホートにも入っていない人たちの表（背景の参照用） | 全員 | ○ `A_bg.csv` |
| `B_i` | あなたのコホート。**加工の入力** | 自チームのみ | ○ `B_practice.csv` |
| `B_self` | `B_i` に真値 `is_rare` を1列足した自己採点用 | 自チームのみ | ○ `B_self.csv` |
| `C_i` | あなたが提出する匿名化済みの表 | 提出物 | — （あなたが作る） |
| `C_k` | 他チーム k の `C`。**攻撃の標的** | 攻撃フェーズで公開 | ○ `targets/C_<k>.csv` |
| 対照 | AIA の答え合わせで「見なくても当たる分」を測るための、標的と条件をそろえた人たち | 事務局のみ | × |
| challenge | AIA で推定する対象の行。標的の人と対照の人が混ざっていて**区別できません** | 全員 | ○ `targets/aia_challenge_<k>.csv` |

配布されるのは **`A_bg` だけ**です（母集団そのものも、対照の名簿も本番では配られません）。

In [1]:
from pathlib import Path
import json
import numpy as np
import pandas as pd

# 配布データの場所を探す（リポジトリ配置でも zip 展開配置でも動くように）
DATA = next(p for p in (Path('..'), Path('../participant_data'), Path('participant_data'))
            if (p / 'B_practice.csv').exists())

for p in sorted(DATA.iterdir()):
    print(('[dir] ' if p.is_dir() else '      ') + p.name)

      A_bg.csv
      B_practice.csv
      B_self.csv
      LICENSE
      README.txt
[dir] sample_submissions
      schema.json
      scoring_config.yaml
[dir] targets
      token.txt
      utility_ref.json


## 1. データの形を見る

1人1行の健診データです。列は次の3種類に分かれます。

- **属性**: `age` `sex` `prefecture` `smoking`
- **検査値**: `BMI` `SBP`（収縮期血圧）`TG`（中性脂肪）`HDL` `ALT`（肝酵素）`FPG`（空腹時血糖）
- **転帰**: `time` `onset` `death`

In [2]:
B = pd.read_csv(DATA / 'B_practice.csv')
print(B.shape)
B.head()

(1049, 14)


,record_id,age,sex,prefecture,BMI,SBP,TG,HDL,ALT,smoking,FPG,time,onset,death
0,0,44,1,神奈川県,25.816369,116.253436,92.804657,59.929361,28.375018,0,113.322461,9.998631,0,0
1,1,47,0,神奈川県,18.656389,112.794305,34.655011,42.034971,7.478586,0,57.735071,9.998631,0,0
2,2,59,1,神奈川県,25.473288,115.495731,67.589276,64.314375,13.711400,1,77.510993,9.998631,0,0
3,3,43,1,神奈川県,22.885859,131.925218,135.663834,51.144123,69.191570,0,89.151418,9.998631,0,0
4,4,55,0,神奈川県,20.975078,119.044400,144.405464,40.752045,5.939557,0,95.215300,8.908966,0,1


### 転帰の3列（ここだけ少し説明が要ります）

この競技のデータは「健診を受けた人たちを最長10年間追いかけた」という形をしています。

- `onset = 1` … 追跡中に**糖尿病を発症した**。`time` はその発症までの年数。
- `onset = 0, death = 1` … 発症しないまま**亡くなった**。`time` は亡くなるまでの年数。
- `onset = 0, death = 0` … 10年間**発症しなかった**。`time` は追跡できた年数（多くは 10）。

最後の2つのように「その人の発症時期は分からない、分かっているのは *ここまでは発症しなかった* ということだけ」
という状態を **打ち切り（censoring）** と呼びます。生存時間解析はこの打ち切りを正しく扱うための道具立てです。

**打ち切りを無視して「発症した人だけ」で平均を取ると答えがずれます。** 加工でこの構造を壊すと
有用性の `U_spec`（結論の一致）が下がります。

In [3]:
print('発症した人      :', int((B.onset == 1).sum()))
print('発症前に死亡    :', int(((B.onset == 0) & (B.death == 1)).sum()))
print('10年発症しなかった:', int(((B.onset == 0) & (B.death == 0)).sum()))
print()
print('→ 大半が「発症しなかった」。この偏りが後で効いてきます（第5節）。')

# 発症の有無で FPG（空腹時血糖）の分布が違うことを見る＝これが再現できるかが U_spec
print()
print(B.groupby('onset')['FPG'].describe()[['count', 'mean', '50%']])

発症した人      : 110
発症前に死亡    : 77
10年発症しなかった: 862

→ 大半が「発症しなかった」。この偏りが後で効いてきます（第5節）。

       count        mean         50%
onset                               
0      939.0   89.539696   89.287679
1      110.0  112.265264  113.739936


## 2. 素朴な匿名化をやってみる

まずは一番簡単な加工＝**年齢を丸めて、検査値に小さなノイズを足す**だけをやってみます。
提出する `C.csv` は **`record_id` を持たない13列**で、**行数は `B` と同じ**です。

In [4]:
rng = np.random.default_rng(0)

C = B.drop(columns=['record_id']).copy()

# (a) 年齢を5歳単位に丸める
C['age'] = (C['age'] // 5 * 5 + 2).clip(40, 74).astype(int)

# (b) 検査値に標準偏差2%のノイズを足す
for col in ['BMI', 'SBP', 'TG', 'HDL', 'ALT', 'FPG']:
    C[col] = C[col] * (1 + rng.normal(0, 0.02, len(C)))

# (c) 値域から出ないように丸め戻す（schema.json の ranges を守らないと提出が棄却されます）
schema = json.loads((DATA / 'schema.json').read_text())
for col, (lo, hi) in schema['ranges'].items():
    C[col] = C[col].clip(lo, hi)

C.to_csv('C.csv', index=False)
print('書き出しました: C.csv', C.shape)
C.head(3)

書き出しました: C.csv (1049, 13)


,age,sex,prefecture,BMI,SBP,TG,HDL,ALT,smoking,FPG,time,onset,death
0,42,1,神奈川県,25.881287,117.084572,89.723976,60.616971,29.069403,0,115.204556,9.998631,0,0
1,47,0,神奈川県,18.607097,113.225163,33.875954,42.454710,7.355143,0,58.588719,9.998631,0,0
2,57,1,神奈川県,25.799561,122.135581,68.993405,65.452668,14.096828,1,78.710884,9.998631,0,0


## 3. 自己採点して「どこが弱いか」を見る

有用性は**手元でサーバと同じ点数**を出せます（Docker が要ります）。

```sh
cd participant_data
docker run --rm -v "$PWD":/w hajimeono/pwscup2026-kit:main-process-20260912 score /w/notebooks/C.csv --dist /w
```

下のセルはそれをノートから実行します（Docker が無ければ飛ばしてください）。

In [5]:
import subprocess

IMAGE = 'hajimeono/pwscup2026-kit:main-process-20260912'   # ★タグを省略しないでください（:latest はフェーズが進むと別のイメージを指します）
FALLBACK = ('→ Docker が使えない場合は starter の環境で同じ点数が出ます:\n'
            f'    cd starter && uv run python -m pwscup2026.kit.selfscore '
            f'../notebooks/C.csv --dist ../participant_data')
cmd = ['docker', 'run', '--rm',
       '-v', f'{DATA.resolve()}:/data',      # 配布データ
       '-v', f'{Path.cwd()}:/out',           # このノートが書いた C.csv
       IMAGE, 'score', '/out/C.csv', '--dist', '/data', '--json']
try:
    out = subprocess.run(cmd, capture_output=True, text=True, timeout=900)
    print(out.stdout[-3000:] if out.returncode == 0 else out.stderr[-1500:])
    if out.returncode != 0:
        print(FALLBACK)
except Exception as e:
    print('Docker を実行できませんでした:', e)
    print(FALLBACK)

failed to connect to the docker API at unix:///var/run/docker.sock; check if the path is correct and if the daemon is running: dial unix /var/run/docker.sock: connect: no such file or directory

→ Docker が使えない場合は starter の環境で同じ点数が出ます:
    cd starter && uv run python -m pwscup2026.kit.selfscore ../notebooks/C.csv --dist ../participant_data


### 読み方

有用性 **U は4つの観点の最小値**です。

| 観点 | 何を見るか |
| --- | --- |
| `U_gen` | 汎用的な統計（周辺分布・識別されにくさ・時間の分布）が再現されているか |
| `U_spec` | **解析の結論が一致するか**（Cox 回帰の係数・予測性能） |
| `U_rare` | **希少な人たち**の特徴が保存されているか |
| `U_valid` | 打ち切りの整合や、あり得ない値の捏造が無いか |

平均ではなく最小値なので、**一番低い観点がそのまま点数**になります。まずそこを直すのが最短です。
リーダーボードにも4つの内訳がそのまま出ます。

### この素朴な加工の結果（参考値）

上のやり方（年齢を5歳丸め＋検査値に2%ノイズ）だと、実測でだいたいこうなります。

| | 値 |
| --- | --- |
| 有用性 U | 0.91 |
| 保護 protection | 0.88 |

**有用性はほとんど落ちていませんが、保護は 1.00 に届きません。**
参考までに、`B` を**一切加工せずそのまま出した**ときの保護も 0.88 です。
つまりこの程度のノイズでは、参照攻撃から見て**何も加工していないのと変わらない**ということです。

守るには「値を少し揺らす」のではなく、**誰か1人の情報がそのまま残らない**ようにする必要があります。
ここが加工の設計の勘どころで、この競技の本題です。

### 提出したら、リーダーボードに載せる（★忘れやすいところです）

CodaBench は、**提出しただけではリーダーボードに載りません。** 提出一覧（My Submissions）の
**Actions 列のいちばん左のアイコン**——マウスを重ねると **Add to Leaderboard** と出ます——を
押してください。載っている提出は、その列が**緑のチェック**に変わります。

**加工・攻撃のどちらのフェーズでも、順位の計算に使われるのは、そのフェーズ終了時点で
リーダーボードに載せている 1 件だけ**です。載せられるのは1件なので、良いものができたら
載せ替えてください。

なお、**1件も載せずにフェーズが終わったチームについては、事務局が最新の有効な提出を
あとから掲載します**（ルールブック §5.3）。とはいえ「最新」が自分のいちばん良い提出とは
限らないので、**出したいものは自分で載せてください。**

## 4. 攻撃をやってみる

攻撃は2種類あり、**1回の提出で両方**出します。

- **MIA（メンバー推定）**: 候補者ごとに「この人は標的チームのコホートに入っていたか」を 0〜1 の確信度で答える → `F_mia.csv`
- **AIA（属性推論）**: challenge の各行について「この人の `time`（発症/追跡年数）は何年か」を答える → `F_aia.csv`

In [6]:
K = 1  # 標的チームの番号（practice では 1〜29 が公開されています）

C_k = pd.read_csv(DATA / 'targets' / f'C_{K}.csv')
chal = pd.read_csv(DATA / 'targets' / f'aia_challenge_{K}.csv')
print('標的の匿名化データ C_k:', C_k.shape)
print('challenge         :', chal.shape, list(chal.columns))
chal.head(3)

標的の匿名化データ C_k: (1049, 13)
challenge         : (326, 8) ['challenge_row_id', 'age', 'SBP', 'TG', 'HDL', 'ALT', 'sex', 'prefecture']


,challenge_row_id,age,SBP,TG,HDL,ALT,sex,prefecture
0,0,63,107.578009,99.788946,40.476891,11.007740,0,埼玉県
1,1,65,145.693750,87.926494,56.238291,14.863065,1,埼玉県
2,2,72,107.982006,46.431071,41.094708,11.726252,0,埼玉県


### 4-1. 素朴な AIA — 似た人の `time` を答える

challenge には `age` `SBP` `TG` `HDL` `ALT` `sex` `prefecture` しかありません（`time` は伏せられています）。
標的の `C_k` から**似た人**を探して、その `time` を答えにしてみます。

In [7]:
def naive_time_hat(chal: pd.DataFrame, C_k: pd.DataFrame) -> np.ndarray:
    """同じ性別・県・年齢帯(±3歳)の人の time の中央値を答えにする素朴な推定。"""
    fallback = float(C_k['time'].median())
    out = np.full(len(chal), fallback)
    for i, row in chal.iterrows():
        m = ((C_k['sex'] == row['sex'])
             & (C_k['prefecture'] == row['prefecture'])
             & (C_k['age'].sub(row['age']).abs() <= 3))
        if m.sum() >= 3:
            out[i] = float(C_k.loc[m, 'time'].median())
    return out

time_hat = naive_time_hat(chal, C_k)
F_aia = pd.DataFrame({'target_k': K, 'challenge_row_id': chal['challenge_row_id'], 'time_hat': time_hat})
F_aia.head()

,target_k,challenge_row_id,time_hat
0,1,0,10.0
1,1,1,10.0
2,1,2,10.0
3,1,3,10.0
4,1,4,10.0


### 4-2. 素朴な MIA — 標的データに「近い人」ほど怪しい

候補者（`B_practice` の人たち）それぞれについて、標的の `C_k` の中で**一番近い行との距離**を測り、
近いほど「この人は入っていた」と強く答えます。

`F_mia.csv` は **行 = 候補者の `record_id`、列 = 標的番号**の表です。ここでは列 `K` だけ埋めます。

In [8]:
from sklearn.neighbors import NearestNeighbors

NUM = ['age', 'BMI', 'SBP', 'TG', 'HDL', 'ALT', 'FPG']

def naive_mia_confidence(cand: pd.DataFrame, C_k: pd.DataFrame) -> np.ndarray:
    """標的データの最近傍までの距離を 0〜1 の確信度に変換する。"""
    mu, sd = C_k[NUM].mean(), C_k[NUM].std().replace(0, 1)
    nn = NearestNeighbors(n_neighbors=1).fit(((C_k[NUM] - mu) / sd).to_numpy())
    d, _ = nn.kneighbors(((cand[NUM] - mu) / sd).to_numpy())
    d = d.ravel()
    return 1.0 / (1.0 + d)   # 近い(d小) → 1に近い

targets = sorted(int(p.stem.split('_')[1]) for p in (DATA / 'targets').glob('C_*.csv'))
F_mia = pd.DataFrame({'record_id': B['record_id']})
for k in targets:
    F_mia[str(k)] = 0.5                      # 未攻撃の標的はとりあえず中立
F_mia[str(K)] = naive_mia_confidence(B, C_k)  # 標的 K だけ本気で答える
F_mia.iloc[:5, :6]

,record_id,1,2,3,4,5
0,0,0.557182,0.5,0.5,0.5,0.5
1,1,0.662634,0.5,0.5,0.5,0.5
2,2,0.434721,0.5,0.5,0.5,0.5
3,3,0.419596,0.5,0.5,0.5,0.5
4,4,0.582910,0.5,0.5,0.5,0.5


In [9]:
# 提出用に書き出す（token.txt と一緒に zip して提出）
F_mia.to_csv('F_mia.csv', index=False)
F_aia.to_csv('F_aia.csv', index=False)
print('書き出しました: F_mia.csv, F_aia.csv')
print('※ 実際の提出では全ての標的について答えるほど得点が伸びます（ここでは1つだけ）。')

書き出しました: F_mia.csv, F_aia.csv
※ 実際の提出では全ての標的について答えるほど得点が伸びます（ここでは1つだけ）。


## 5. 採点の考え方 —「対照と比べて超過した分」だけを数える

AIA の得点は**当たった率そのままではありません**。理由は第1節で見たとおり、
**大半の人は10年間発症しません**。だから「全員 `time = 10`」と答えるだけで8割くらい当たってしまいます。
これは標的のデータを見なくても言えることで、**攻撃の成果ではありません**。

そこで challenge には、標的の人（**member**）と、条件をそろえた**対照（control）**が混ぜてあります。
どちらがどちらかは攻撃者には分かりません。答え合わせのときだけ事務局が2群に分けて、

- `p_m` … 標的の人での的中率
- `p_c` … 対照での的中率 ＝ **見なくても当たる分（下駄）**

を出し、**下駄を超えた分**だけを得点にします。

$$R = \frac{p_m - p_c}{1 - p_c}$$

In [10]:
def R_from_rate(p_m: float, p_c: float) -> float:
    """対照で当たる分を差し引き、残っていた余地で割る。"""
    if p_c >= 1.0:
        return 0.0
    return max(0.0, (p_m - p_c) / (1.0 - p_c))

# 数値例: challenge 100行（標的50・対照50）。標的で30行、対照で25行を ±0.5年以内に当てた。
p_m, p_c = 30 / 50, 25 / 50
print(f'p_m = {p_m:.2f}  （標的での的中率）')
print(f'p_c = {p_c:.2f}  （対照でも当たる＝下駄）')
print(f'残っていた余地 = 1 - p_c = {1 - p_c:.2f}')
print(f'R   = ({p_m:.2f} - {p_c:.2f}) / {1 - p_c:.2f} = {R_from_rate(p_m, p_c):.2f}')
print()
for pm, pc, note in [(1.00, 0.50, '元データそのまま（ほぼ全部当たる）'),
                     (0.50, 0.50, '対照と同じ精度＝何も分かっていない'),
                     (0.85, 0.80, '的中率85%でも、下駄が80%なら得点は0.25にしかならない')]:
    print(f'  p_m={pm:.2f} p_c={pc:.2f} → R={R_from_rate(pm, pc):.2f}   {note}')

p_m = 0.60  （標的での的中率）
p_c = 0.50  （対照でも当たる＝下駄）
残っていた余地 = 1 - p_c = 0.50
R   = (0.60 - 0.50) / 0.50 = 0.20

  p_m=1.00 p_c=0.50 → R=1.00   元データそのまま（ほぼ全部当たる）
  p_m=0.50 p_c=0.50 → R=0.00   対照と同じ精度＝何も分かっていない
  p_m=0.85 p_c=0.80 → R=0.25   的中率85%でも、下駄が80%なら得点は0.25にしかならない


あなたが攻撃を提出すると、**詳細結果にこの `p_m` と `p_c` が実際の値で表示されます**。
上の関数に入れれば、得点の内訳をおおよそ確かめられます。ただし表示される `p_m`・`p_c` は相手チームごとの的中率を平均した値で、AIA攻撃力は相手ごとに求めた R を平均した値です。平均してから R を求めるのと、R を求めてから平均するのは同じにならないため、**この式で得られるのは目安**です。「そもそも当てられていないのか、それとも下駄 `p_c` が高いだけなのか」を読み分けるための値としてお使いください。

MIA も考え方は同じで、「どれだけ自信をもって当てたか」を **偽陽性1%という厳しい条件のもとでの的中率**
（TPR@FPR=1%）で測ります。全員に同じ確信度を出すと順位がつかず、得点になりません。

## 6. 次の一歩

- **守る側**: 自己採点で一番低い観点を見つけて、そこを狙って加工を変える。`U_spec` が低いときは
  発症との関係（例：`FPG` が高い人ほど発症しやすい）が壊れていないか疑ってください。
- **攻める側**: ここでは1つの標的にしか答えていません。全標的に答え、`time_hat` の精度を上げてください。
  **対照でも当たる答え方（全員10年など）では得点になりません。**
- **採点の中身を開けて見たい**: 同じフォルダの **`採点のしくみ.ipynb`** で、4観点それぞれが何を測っていて
  どの操作で点が動くのかを、実際に加工を作りながら確かめられます。
- 提出物の正確な仕様は **ルールブック**と **`participant_data/sample_submissions/`** を確認してください。

うまくいかないときは CodaBench の FAQ ページ、または事務局までご連絡ください。